In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler

DATA_DIR = "../input/leaf-classification"

train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")
sample_path = os.path.join(DATA_DIR, "sample_submission.csv")

train_csv = pd.read_csv(train_path)
test_csv = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path)

train_csv.head()



,id,species,margin1,margin2,margin3,margin4,margin5,margin6,margin7,margin8,...,texture55,texture56,texture57,texture58,texture59,texture60,texture61,texture62,texture63,texture64
0,186,Alnus_Cordata,0.001953,0.005859,0.039062,0.019531,0.025391,0.001953,0.058594,0.0,...,0.101560,0.000000,0.006836,0.000000,0.008789,0.000000,0.00000,0.008789,0.003906,0.023438
1,1152,Tilia_Platyphyllos,0.001953,0.011719,0.023438,0.033203,0.056641,0.003906,0.031250,0.0,...,0.007812,0.003906,0.002930,0.010742,0.046875,0.000977,0.00000,0.018555,0.008789,0.040039
2,435,Quercus_Imbricaria,0.037109,0.052734,0.031250,0.017578,0.003906,0.111330,0.029297,0.0,...,0.079102,0.003906,0.002930,0.000977,0.005859,0.000000,0.03125,0.004883,0.002930,0.035156
3,1467,Acer_Platanoids,0.007812,0.013672,0.072266,0.007812,0.007812,0.056641,0.003906,0.0,...,0.127930,0.041016,0.006836,0.003906,0.024414,0.000000,0.00000,0.017578,0.000000,0.025391
4,1222,Quercus_Semecarpifolia,0.027344,0.062500,0.060547,0.005859,0.001953,0.064453,0.029297,0.0,...,0.045898,0.000000,0.059570,0.003906,0.045898,0.000000,0.00000,0.002930,0.000000,0.034180


In [2]:
feature_cols = [c for c in train_csv.columns if c not in ("id", "species")]
xtrain = train_csv[feature_cols].copy()

if "id" not in test_csv.columns:
    raise ValueError("test.csv is missing required column 'id'")
testid = test_csv["id"]
xtest_raw = test_csv.drop(columns=["id"])

# Keep the strict train/test feature match to avoid silent garbage predictions.
missing_in_test = [c for c in feature_cols if c not in xtest_raw.columns]
extra_in_test = [c for c in xtest_raw.columns if c not in feature_cols]
if missing_in_test or extra_in_test:
    raise ValueError(
        "Train/test feature mismatch.\n"
        f"Missing in test (present in train): {missing_in_test}\n"
        f"Extra in test (not in train): {extra_in_test}\n"
        "Fixing this is required for meaningful predictions."
    )

xtest = xtest_raw[feature_cols].copy()

scaler = MinMaxScaler()
xtrain_scaled = pd.DataFrame(
    scaler.fit_transform(xtrain), columns=feature_cols, index=xtrain.index
)
xtest_scaled = pd.DataFrame(
    scaler.transform(xtest), columns=feature_cols, index=xtest.index
)

xtrain_scaled.head()



,margin1,margin2,margin3,margin4,margin5,margin6,margin7,margin8,margin9,margin10,...,texture55,texture56,texture57,texture58,texture59,texture60,texture61,texture62,texture63,texture64
0,0.023254,0.028569,0.281690,0.114942,0.228070,0.006289,0.638300,0.0,0.000000,0.320001,...,0.236356,0.000000,0.039549,0.000000,0.082565,0.00000,0.000000,0.023376,0.044941,0.165523
1,0.023254,0.057144,0.169020,0.195404,0.508767,0.012578,0.340425,0.0,0.076918,0.359998,...,0.018181,0.019322,0.016951,0.053656,0.440348,0.00169,0.000000,0.049351,0.101123,0.282761
2,0.441858,0.257139,0.225355,0.103449,0.035085,0.358493,0.319150,0.0,0.000000,0.240006,...,0.184091,0.019322,0.016951,0.004880,0.055040,0.00000,0.206448,0.012987,0.033711,0.248277
3,0.093018,0.066667,0.521137,0.045975,0.070170,0.182389,0.042550,0.0,0.589744,0.000000,...,0.297726,0.202899,0.039549,0.019510,0.229347,0.00000,0.000000,0.046752,0.000000,0.179315
4,0.325586,0.304759,0.436627,0.034481,0.017542,0.207545,0.319150,0.0,0.000000,0.300002,...,0.106817,0.000000,0.344634,0.019510,0.431170,0.00000,0.000000,0.007793,0.000000,0.241384


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.decomposition import PCA



In [4]:
from sklearn.linear_model import LogisticRegression

ytrain = train_csv["species"].astype(str).to_numpy()

logic = LogisticRegression(
    max_iter=2000,
    multi_class="multinomial",
    solver="lbfgs",
    C=0.05,
    random_state=0,
)

logic.fit(xtrain_scaled, ytrain)

Ytest = logic.predict_proba(xtest_scaled)
print(Ytest.shape)



(99, 99)


In [5]:
# Change rationale: the competition metric is logloss, so any class/probability misalignment
# massively worsens score. We therefore (1) build probabilities in model class order first,
# then (2) reorder by sample submission column names by *name*, with strict validation that
# the class sets match (as they should in this competition), to avoid silent catastrophic errors.
eps = 1e-15

sample_cols = list(sample_sub.columns)
if len(sample_cols) < 2 or sample_cols[0] != "id":
    raise ValueError("sample_submission.csv must have 'id' as the first column.")
expected_class_cols = sample_cols[1:]

model_class_cols = list(map(str, logic.classes_))

train_class_set = set(map(str, train_csv["species"].astype(str).unique()))
sample_class_set = set(map(str, expected_class_cols))
model_class_set = set(model_class_cols)

# Fail fast if the environment's sample submission is inconsistent; otherwise any mapping
# can yield extremely poor logloss (like your current 3.7).
if train_class_set != sample_class_set:
    missing_from_sample = sorted(list(train_class_set - sample_class_set))
    extra_in_sample = sorted(list(sample_class_set - train_class_set))
    raise ValueError(
        "Sample submission class columns do not match training species labels.\n"
        f"Missing from sample (present in train): {missing_from_sample}\n"
        f"Extra in sample (not in train): {extra_in_sample}\n"
        "This must match for a valid submission schema in this competition."
    )

if model_class_set != sample_class_set:
    missing_from_model = sorted(list(sample_class_set - model_class_set))
    extra_in_model = sorted(list(model_class_set - sample_class_set))
    raise ValueError(
        "Model classes do not match sample submission classes.\n"
        f"Missing from model: {missing_from_model}\n"
        f"Extra in model: {extra_in_model}\n"
    )

# Build prediction dataframe in the exact order returned by predict_proba (logic.classes_).
proba_values = np.clip(Ytest.astype(float), eps, 1.0 - eps)
pred_df = pd.DataFrame(proba_values, columns=model_class_cols)

# Reorder columns to exactly match sample submission (by name).
pred_df = pred_df.loc[:, expected_class_cols]

# Optional row-normalization for coherence; Kaggle rescales anyway, but this keeps probabilities well-formed.
prob_mat = pred_df.to_numpy(dtype=float)
row_sums = prob_mat.sum(axis=1, keepdims=True)
row_sums = np.where(row_sums <= 0, 1.0, row_sums)
prob_mat = prob_mat / row_sums
prob_mat = np.clip(prob_mat, eps, 1.0 - eps)
pred_df.loc[:, :] = prob_mat

sub = pd.concat([pd.DataFrame({"id": testid.values}), pred_df], axis=1)
sub = sub.loc[:, sample_cols]

if sub.shape[0] != test_csv.shape[0]:
    raise ValueError("Row count mismatch between predictions and test set.")
if list(sub.columns) != sample_cols:
    raise ValueError("Submission columns do not match sample_submission.csv exactly.")

sub.to_csv("submission.csv", index=False)
sub.head()

,id,Acer_Capillipes,Acer_Circinatum,Acer_Mono,Acer_Opalus,Acer_Palmatum,Acer_Pictum,Acer_Platanoids,Acer_Rubrum,Acer_Rufinerve,...,Salix_Fragilis,Salix_Intergra,Sorbus_Aria,Tilia_Oliveri,Tilia_Platyphyllos,Tilia_Tomentosa,Ulmus_Bergmanniana,Viburnum_Tinus,Viburnum_x_Rhytidophylloides,Zelkova_Serrata
0,1202,0.007439,0.009424,0.011900,0.012150,0.007455,0.009478,0.009960,0.009709,0.007702,...,0.008229,0.010161,0.007247,0.013707,0.006977,0.010854,0.008582,0.010868,0.010153,0.007884
1,992,0.009927,0.010713,0.008168,0.010021,0.012558,0.009039,0.009500,0.009733,0.009771,...,0.014838,0.008665,0.019039,0.013811,0.014335,0.008881,0.013121,0.011250,0.013234,0.007631
2,491,0.008519,0.009826,0.007944,0.011140,0.008219,0.009094,0.006497,0.008927,0.008813,...,0.006133,0.008197,0.008483,0.009809,0.008930,0.011435,0.007590,0.007984,0.010205,0.008284
3,1201,0.008006,0.007949,0.017178,0.013862,0.006724,0.007541,0.011822,0.006741,0.006618,...,0.009583,0.026466,0.005650,0.010225,0.006014,0.009148,0.007545,0.017108,0.007969,0.007971
4,72,0.009353,0.009583,0.015693,0.013493,0.006385,0.010880,0.014569,0.007358,0.007680,...,0.014988,0.012565,0.004803,0.005497,0.004041,0.011514,0.010996,0.015711,0.005896,0.006057
